# Qwen Text And Image Smoke Test

This notebook calls Qwen directly without going through the local server.

- Text: OpenAI-compatible `chat/completions`
- Image: DashScope multimodal image generation endpoint
- Secrets: loaded from `.env.local`, then `.env`, then process environment
- Region: defaults to `intl`, but you can switch to `us` or `china` in the config cell
- Output files: saved to `artifacts/qwen_notebook_smoke`

Before running it, fill `QWEN_API_KEY=` in `.env.local` or `.env`.
If your key was created outside Singapore international region, change `REGION_PRESET` in the config cell or override the base URLs manually.
If you want to try other DashScope image models, change `IMAGE_MODEL` to `wan2.7-image-pro` or `z-image-turbo` in the config cell.


In [ ]:
from __future__ import annotations

import json
import mimetypes
import os
from datetime import datetime
from pathlib import Path

import requests
from dotenv import load_dotenv
from IPython.display import Image as IPyImage, display


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "package.json").exists() and (candidate / "test_ipynb").exists():
            return candidate
    raise RuntimeError("Could not locate the repo root from the current notebook path.")


def first_non_empty(*values: str | None) -> str | None:
    for value in values:
        if value and value.strip():
            return value.strip()
    return None


def mask_secret(value: str | None) -> str:
    if not value:
        return "<missing>"
    if len(value) <= 8:
        return "*" * len(value)
    return f"{value[:4]}...{value[-4:]}"


repo_root = find_repo_root()
for env_file in [repo_root / ".env.local", repo_root / ".env"]:
    if env_file.exists():
        load_dotenv(env_file, override=False)

output_dir = repo_root / "artifacts" / "qwen_notebook_smoke"
output_dir.mkdir(parents=True, exist_ok=True)

REGION_PRESET = os.getenv("TRPG_QWEN_REGION", "intl").strip().lower()
REGION_ENDPOINTS = {
    "us": {
        "text": "https://dashscope-us.aliyuncs.com/compatible-mode/v1",
        "image": "https://dashscope-us.aliyuncs.com/api/v1",
    },
    "intl": {
        "text": "https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
        "image": "https://dashscope-intl.aliyuncs.com/api/v1",
    },
    "china": {
        "text": "https://dashscope.aliyuncs.com/compatible-mode/v1",
        "image": "https://dashscope.aliyuncs.com/api/v1",
    },
}
region_endpoints = REGION_ENDPOINTS.get(REGION_PRESET, REGION_ENDPOINTS["intl"])

API_KEY = first_non_empty(
    os.getenv("QWEN_API_KEY"),
    os.getenv("TRPG_QWEN_API_KEY"),
    os.getenv("DASHSCOPE_API_KEY"),
    os.getenv("BAILIAN_API_KEY"),
)
TEXT_BASE_URL = os.getenv("TRPG_QWEN_BASE_URL", region_endpoints["text"]).rstrip("/")
TEXT_MODEL = os.getenv("TRPG_QWEN_MODEL", "qwen-plus")
IMAGE_BASE_URL = first_non_empty(
    os.getenv("TRPG_QWEN_IMAGE_BASE_URL"),
    os.getenv("TRPG_DASHSCOPE_IMAGE_BASE_URL"),
    region_endpoints["image"],
).rstrip("/")
IMAGE_MODEL = os.getenv("TRPG_QWEN_IMAGE_MODEL", "qwen-image-2.0-pro")
REQUEST_TIMEOUT = int(os.getenv("TRPG_QWEN_NOTEBOOK_TIMEOUT", "180"))

TEXT_SYSTEM_PROMPT = "You are a concise assistant for an AI TRPG project."
TEXT_USER_PROMPT = (
    "Write a short opening narration for a campus mystery TRPG scene. "
    "Keep it under 120 words and end with a subtle hook."
)
IMAGE_PROMPT = (
    "A dramatic manga-style comic panel set in a modern university corridor at dusk, "
    "a student detective pauses beside a classroom door, warm sunset light, strong composition, "
    "clean linework, cinematic atmosphere"
)
IMAGE_NEGATIVE_PROMPT = "blurry, low detail, extra fingers, watermark, logo, unreadable text"
IMAGE_SIZE = None
WATERMARK = False

if not API_KEY:
    raise RuntimeError(
        "Missing Qwen API key. Fill QWEN_API_KEY= in .env.local or .env before running this notebook."
    )

print("Repo root:", repo_root)
print("Output dir:", output_dir)
print("Region preset:", REGION_PRESET)
print("Text endpoint:", TEXT_BASE_URL)
print("Text model:", TEXT_MODEL)
print("Image endpoint:", IMAGE_BASE_URL)
print("Image model:", IMAGE_MODEL)
print("API key:", mask_secret(API_KEY))


In [ ]:
def timestamp_slug() -> str:
    return datetime.now().strftime("%Y%m%d_%H%M%S")


def save_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def parse_json_response(response: requests.Response) -> dict:
    try:
        return response.json()
    except requests.JSONDecodeError as exc:
        raise RuntimeError(
            f"Expected JSON but got HTTP {response.status_code}: {response.text[:500]}"
        ) from exc


def build_region_hint(endpoint: str) -> str:
    if "dashscope-us.aliyuncs.com" in endpoint:
        return (
            "Current endpoint is US (Virginia). If your key was created in Singapore or Beijing, "
            "switch REGION_PRESET to 'intl' or 'china', or override TRPG_QWEN_BASE_URL / TRPG_QWEN_IMAGE_BASE_URL."
        )
    if "dashscope-intl.aliyuncs.com" in endpoint:
        return (
            "Current endpoint is International (Singapore). If your key was created in US (Virginia) or Beijing, "
            "switch REGION_PRESET to 'us' or 'china', or override TRPG_QWEN_BASE_URL / TRPG_QWEN_IMAGE_BASE_URL."
        )
    if "dashscope.aliyuncs.com" in endpoint:
        return (
            "Current endpoint is China (Beijing). If your key was created in US (Virginia) or Singapore, "
            "switch REGION_PRESET to 'us' or 'intl', or override TRPG_QWEN_BASE_URL / TRPG_QWEN_IMAGE_BASE_URL."
        )
    return "Check that the API key and endpoint belong to the same Model Studio region and workspace."


def raise_for_api_error(response: requests.Response, payload: dict, fallback_message: str, endpoint: str) -> None:
    if response.ok:
        return
    message = (
        payload.get("error", {}).get("message")
        if isinstance(payload.get("error"), dict)
        else payload.get("message")
    )
    final_message = message or f"{fallback_message} (HTTP {response.status_code})"
    if "Incorrect API key provided" in final_message or "Invalid API-key" in final_message:
        final_message = f"{final_message}\nEndpoint: {endpoint}\nHint: {build_region_hint(endpoint)}"
    raise RuntimeError(final_message)


def call_qwen_text(system_prompt: str, user_prompt: str) -> dict:
    request_body = {
        "model": TEXT_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": 0.7,
    }
    response = requests.post(
        f"{TEXT_BASE_URL}/chat/completions",
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {API_KEY}",
        },
        json=request_body,
        timeout=REQUEST_TIMEOUT,
    )
    payload = parse_json_response(response)
    raise_for_api_error(response, payload, "Qwen text request failed", f"{TEXT_BASE_URL}/chat/completions")

    text = payload["choices"][0]["message"]["content"].strip()
    stem = output_dir / f"{timestamp_slug()}_qwen_text"
    save_json(stem.with_suffix(".request.json"), request_body)
    save_json(stem.with_suffix(".response.json"), payload)
    stem.with_suffix(".txt").write_text(text, encoding="utf-8")
    return {
        "text": text,
        "request_path": stem.with_suffix(".request.json"),
        "response_path": stem.with_suffix(".response.json"),
        "text_path": stem.with_suffix(".txt"),
    }


def build_qwen_image_parameters() -> dict:
    parameters = {"watermark": WATERMARK}
    if IMAGE_SIZE:
        parameters["size"] = IMAGE_SIZE
    if IMAGE_NEGATIVE_PROMPT and IMAGE_NEGATIVE_PROMPT.strip():
        parameters["negative_prompt"] = IMAGE_NEGATIVE_PROMPT.strip()

    normalized_model = IMAGE_MODEL.strip().lower()
    if normalized_model.startswith("wan2.7-image"):
        parameters["thinking_mode"] = True
        parameters["n"] = 1
    elif normalized_model == "z-image-turbo":
        parameters["prompt_extend"] = False
    else:
        parameters["prompt_extend"] = True

    return parameters


def call_qwen_image(prompt: str) -> dict:
    request_body = {
        "model": IMAGE_MODEL,
        "input": {
            "messages": [
                {
                    "role": "user",
                    "content": [{"text": prompt}],
                }
            ]
        },
        "parameters": build_qwen_image_parameters(),
    }
    response = requests.post(
        f"{IMAGE_BASE_URL}/services/aigc/multimodal-generation/generation",
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {API_KEY}",
        },
        json=request_body,
        timeout=REQUEST_TIMEOUT,
    )
    payload = parse_json_response(response)
    raise_for_api_error(response, payload, "Qwen image request failed", f"{IMAGE_BASE_URL}/services/aigc/multimodal-generation/generation")

    content_items = payload.get("output", {}).get("choices", [{}])[0].get("message", {}).get("content", [])
    revised_prompt = next(
        (item.get("text", "").strip() for item in content_items if isinstance(item, dict) and item.get("text")),
        None,
    )
    image_url = next(
        (item.get("image") for item in content_items if isinstance(item, dict) and item.get("image")),
        None,
    )
    if not image_url:
        raise RuntimeError(f"Qwen image response did not contain an image URL: {json.dumps(payload, ensure_ascii=False)[:800]}")

    image_response = requests.get(image_url, timeout=REQUEST_TIMEOUT)
    image_response.raise_for_status()
    mime_type = image_response.headers.get("content-type", "image/png").split(";")[0].strip()
    extension = mimetypes.guess_extension(mime_type) or ".png"

    stem = output_dir / f"{timestamp_slug()}_qwen_image"
    image_path = stem.with_suffix(extension)
    image_path.write_bytes(image_response.content)
    save_json(stem.with_suffix(".request.json"), request_body)
    save_json(stem.with_suffix(".response.json"), payload)
    return {
        "image_path": image_path,
        "image_url": image_url,
        "revised_prompt": revised_prompt,
        "request_path": stem.with_suffix(".request.json"),
        "response_path": stem.with_suffix(".response.json"),
        "mime_type": mime_type,
    }


In [ ]:
text_result = call_qwen_text(TEXT_SYSTEM_PROMPT, TEXT_USER_PROMPT)
print(text_result["text"])
print()
print("Saved text request:", text_result["request_path"])
print("Saved text response:", text_result["response_path"])
print("Saved text output:", text_result["text_path"])


In [ ]:
image_result = call_qwen_image(IMAGE_PROMPT)
print("Image URL:", image_result["image_url"])
print("Revised prompt:", image_result["revised_prompt"])
print("Saved image:", image_result["image_path"])
print("Saved image request:", image_result["request_path"])
print("Saved image response:", image_result["response_path"])
display(IPyImage(filename=str(image_result["image_path"])))
